# 総研大 2024年度 講義「シミュレーション天文学」 / SOKEN-dai FY2024 Lecture Course: “Simulation Astronomy”
### 担当：馬場淳一 (国立天文台 科学研究部/鹿児島大学 天の川銀河研究センター) / Junichi Baba (D. of Science, NAOJ / Kagoshima University)

This code is provided for educational purposes only. Users are encouraged to utilize it at their own risk. I take no responsibility for any results obtained from research or analyses conducted using this code. It is essential to deepen your understanding and verify the results independently. Additionally, it is recommended to cross-reference with other reliable resources to ensure the appropriate methodologies are applied.

このコードは教育目的で提供されています。使用にあたっては、自己責任で行ってください。提供されたコードを基に行った研究や解析によって得られた結果について、私は一切の責任を負いません。使用する際には、必ず自身の理解を深め、結果を検証してください。また、他の信頼できるリソースと照らし合わせて、適切な方法論を選択することをお勧めします。

## 学習目標と計算上の前提

円盤上の粒子に3次元Newton重力を適用します。返り値は加速度ではなく力 F_i=m_i a_i です。自己粒子を含む内部ノードは必ず開きます。softening無しの教育例であり、同位置粒子は未定義として拒否します。θ=0 は直接和と一致し、一般のθでは粒子ごとの相対誤差を評価します。

検証課題と判定基準は [演習ガイド](../docs/exercises.md) を参照してください。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Plot configuration parameters
parameters = {
    'axes.labelsize': 26,  # Set font size of x and y axis labels to 26
    'xtick.labelsize': 24,  # Set font size of x-axis ticks to 24
    'ytick.labelsize': 24,  # Set font size of y-axis ticks to 24
    'axes.titlesize': 24,   # Set font size of individual axis titles to 24
    'figure.titlesize': 24, # Set font size of the overall figure title to 24
    'legend.fontsize': 22,  # Set font size of the legend to 22
    'figure.figsize': [10, 6],  # Set default figure size to 10 inches by 6 inches
    'axes.linewidth': 2.0  # Set the line width of axis borders to 2.0
}

# Apply the parameters to matplotlib's default settings
plt.rcParams.update(parameters)
np.random.seed(20241108)  # Reproducible teaching examples


In [ ]:
G = 1  # 重力定数（適切な値を設定）

In [ ]:
class Body:
    def __init__(self, position, mass):
        self.position = np.array(position)
        self.mass = mass
        
class QuadNode:
    def __init__(self, boundary, depth=0):
        self.depth = depth
        self.boundary = boundary  # Boundary of the node (x_min, x_max, y_min, y_max)
        self.mass = 0
        self.center_of_mass = np.zeros(2)
        self.body = None
        self.children = []
        self.is_divided = False  # Records whether this node has been divided

    def insert(self, body):
        if not self.contains(body.position):
            return False

        # If this node is empty, place the body here
        if self.body is None and len(self.children) == 0:
            self.body = body
            self.mass += body.mass
            self.center_of_mass = body.position
            return True

        # If this node contains a body, subdivide
        if self.body is not None:
            if self.depth >= 64:
                raise ValueError("Maximum tree depth exceeded")
            self.subdivide()
            # Reinsert the existing body into the correct child
            self.children[self.get_child_index(self.body.position)].insert(self.body)

            # Clear the body, as it has been moved to children
            self.body = None
            self.is_divided = True  # Record that the node has been divided

        # Insert the new body into the correct child
        self.children[self.get_child_index(body.position)].insert(body)

        # Update center of mass
        self.update_center_of_mass(body)
        return True

    def get_child_index(self, position):
        x_min, x_max, y_min, y_max = self.boundary
        mid_x = (x_min + x_max) / 2
        mid_y = (y_min + y_max) / 2
        
        if position[0] < mid_x:
            if position[1] < mid_y:
                return 0  # NW
            else:
                return 2  # SW
        else:
            if position[1] < mid_y:
                return 1  # NE
            else:
                return 3  # SE

    def subdivide(self):
        x_min, x_max, y_min, y_max = self.boundary
        mid_x = (x_min + x_max) / 2
        mid_y = (y_min + y_max) / 2

        self.children = [
            QuadNode((x_min, mid_x, y_min, mid_y), self.depth + 1),  # NW
            QuadNode((mid_x, x_max, y_min, mid_y), self.depth + 1),  # NE
            QuadNode((x_min, mid_x, mid_y, y_max), self.depth + 1),  # SW
            QuadNode((mid_x, x_max, mid_y, y_max), self.depth + 1)   # SE
        ]

    def contains(self, position):
        x_min, x_max, y_min, y_max = self.boundary
        return x_min <= position[0] <= x_max and y_min <= position[1] <= y_max

    def update_center_of_mass(self, body):
        total_mass = self.mass + body.mass
        if self.mass == 0:
            self.center_of_mass = body.position
        else:
            self.center_of_mass = (self.center_of_mass * self.mass + body.position * body.mass) / total_mass
        self.mass = total_mass

In [ ]:
# BarnesHut Class
class BarnesHut:
    def __init__(self, boundary):
        self._positions = set()
        self.root = QuadNode(boundary)

    def insert(self, body):
        if not self.root.contains(body.position):
            raise ValueError("Particle outside root boundary")
        position = tuple(body.position)
        if position in self._positions:
            raise ValueError("Coincident particles are unsupported without softening")
        if not self.root.insert(body):
            raise ValueError("Tree insertion failed (maximum depth)")
        self._positions.add(position)

    def calculate_gravitational_force(self, body, theta=0.5):
        return self._calculate_force(self.root, body, theta)

    def _calculate_force(self, node, body, theta):
        distance = np.linalg.norm(body.position - node.center_of_mass)
    
        # Calculate the size of the node
        x_min, x_max, y_min, y_max = node.boundary
        node_size = max(x_max - x_min, y_max - y_min)
    
        if len(node.children) == 0 or (not node.contains(body.position) and distance > 0 and node_size / distance < theta):
            # Use the node as an approximation
            force = self.calculate_force_between(body, node)
            return force
        else:
            force = np.zeros(2)
            for child in node.children:
                if child:
                    force += self._calculate_force(child, body, theta)
            return force

    def calculate_force_between(self, body, node):
        r = node.center_of_mass - body.position
        distance = np.linalg.norm(r)
        if distance == 0:
            return np.zeros(2)  # Return zero force if the body and center of mass are at the same position
        force_magnitude = G * body.mass * node.mass / distance**3
        force = force_magnitude * r
        return force
        
    def visualize_tree(self, node, ax):
        # Draw the boundary of the node
        x_min, x_max, y_min, y_max = node.boundary
        ax.add_patch(plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, fill=None, edgecolor='black'))

        # If a body exists, draw its position
        if node.body is not None:
            ax.scatter(node.body.position[0], node.body.position[1], color='blue', s=100)

        # If there are child nodes, draw them recursively
        for child in node.children:
            if child:
                self.visualize_tree(child, ax)

In [ ]:
# Function to calculate gravitational force directly between all bodies
def calculate_gravitational_force_direct(bodies, G=1):
    forces = []
    for i, body_i in enumerate(bodies):
        total_force = np.zeros(2)  # Initialize gravitational force
        for j, body_j in enumerate(bodies):
            if i != j:  # Do not apply force to itself
                r = body_j.position - body_i.position  # Distance vector
                distance = np.linalg.norm(r)  # Distance
                if distance != 0:
                    force_magnitude = G * body_i.mass * body_j.mass / distance**3
                    total_force += force_magnitude * r  # Add force in the direction of the vector
        forces.append(total_force)
    return forces

In [ ]:
# Configuration
Mtot = 10  # Total mass
num_bodies = 50  # Number of particles
R = 5  # Radius of the disk

# Generate random positions
# Generate random angles theta and distances r for each particle
theta = np.random.uniform(0, 2 * np.pi, num_bodies)  # Random angles (0 to 2π)
r = np.sqrt(np.random.uniform(0, R**2, num_bodies))  # Random radii (0 to R) - uniform distribution using √(U)

# Convert from polar to Cartesian coordinates
x_positions = r * np.cos(theta)
y_positions = r * np.sin(theta)

# Create Body instances (assuming equal mass of Mtot / num_bodies)
mass = Mtot / num_bodies
bodies = [Body(position=(x, y), mass=mass) for x, y in zip(x_positions, y_positions)]

# Create a Barnes-Hut tree
boundary = (-R, R, -R, R)
bh_tree = BarnesHut(boundary)

# Insert particles into the tree
for body in bodies:
    bh_tree.insert(body)

# Calculate gravitational forces using the tree method
forces_tree = []
for body in bodies:
    force = bh_tree.calculate_gravitational_force(body, theta=0.2)
    forces_tree.append(force)

# Calculate gravitational forces directly between all particles
forces_direct = calculate_gravitational_force_direct(bodies)

# Calculate the differences between the tree method and direct calculation
forces_diff = []
for i in range(num_bodies):
    direct_norm = np.linalg.norm(forces_direct[i])  # Magnitude (norm) of the direct calculation vector
    if direct_norm != 0:
        normalized_diff = (forces_tree[i] - forces_direct[i]) / direct_norm
    else:
        normalized_diff = forces_tree[i] - forces_direct[i]  # Keep as is if the norm is zero
    forces_diff.append(normalized_diff)

# Calculate the magnitudes of the difference vectors
forces_diff_magnitudes = [100 * np.linalg.norm(force_diff) for force_diff in forces_diff]
median_value = np.median(forces_diff_magnitudes)
print(median_value)

#####################
# Plot the spatial distribution
size = 8
plt.figure(figsize=(2 * size, size))  # Widen for three plots

# Left panel: Comparison of tree method and direct calculation
plt.subplot(1, 2, 1)
bh_tree.visualize_tree(bh_tree.root, plt.gca())
plt.scatter([body.position[0] for body in bodies], [body.position[1] for body in bodies], s=100, color='blue', label='Particles')

# Draw gravitational force vectors
# scale_factor = 2  # Adjust as needed
# Plot results from the tree method in red
# for i, body in enumerate(bodies):
#     plt.quiver(body.position[0], body.position[1], forces_tree[i][0] * scale_factor, forces_tree[i][1] * scale_factor,
#                angles='xy', scale_units='xy', scale=1, color='red', label='Tree Force' if i == 0 else "")
# Plot results from direct calculation in green
# for i, body in enumerate(bodies):
#     plt.quiver(body.position[0], body.position[1], forces_direct[i][0] * scale_factor, forces_direct[i][1] * scale_factor,
#                angles='xy', scale_units='xy', scale=1, color='green', label='Direct Force' if i == 0 else "")

plt.xlim(-R, R)
plt.ylim(-R, R)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Tree vs Direct Gravitational Force')
plt.grid()
plt.legend()
plt.axis('equal')

# Right panel: Histogram of the magnitudes of difference vectors
plt.subplot(1, 2, 2)
plt.hist(forces_diff_magnitudes, bins=20, color='purple', edgecolor='black')
plt.axvline(median_value, color='red', linestyle='dashed', linewidth=2, label=f'Median: {median_value:.2f}')
plt.xlabel('|Tree - Direct|/|Direct| [%]')
plt.ylabel('Frequency')
plt.title('Histogram')

plt.tight_layout()
plt.show()

In [ ]:
# Configuration
Mtot = 10  # Total mass
num_bodies = 10  # Number of particles
R = 5  # Radius of the disk

# Generate random positions
theta = np.random.uniform(0, 2 * np.pi, num_bodies)  # Random angles (0 to 2π)
r = np.sqrt(np.random.uniform(0, R**2, num_bodies))  # Random radii (0 to R)

# Convert from polar to Cartesian coordinates
x_positions = r * np.cos(theta)
y_positions = r * np.sin(theta)

# Create Body instances (assuming equal mass of Mtot / num_bodies)
mass = Mtot / num_bodies
bodies = [Body(position=(x, y), mass=mass) for x, y in zip(x_positions, y_positions)]

# Create a Barnes-Hut tree
boundary = (-R, R, -R, R)
bh_tree = BarnesHut(boundary)

# Insert particles into the tree
for body in bodies:
    bh_tree.insert(body)

# Function to visualize the tree structure recursively
def visualize_tree_levels(node, ax, level, current_level=0):
    if current_level > level:
        return
    
    # Draw the boundary of the node
    x_min, x_max, y_min, y_max = node.boundary
    if node.is_divided:
        color = 'red'
        lw = 3
    else:
        color = 'black'
        lw = 2
    ax.add_patch(plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min, fill=None, lw=lw, edgecolor=color))

    # If a body exists, draw its position
    if node.body is not None:
        ax.scatter(node.body.position[0], node.body.position[1], color='blue', s=100)

    # If there are child nodes, draw them recursively
    if node.children:
        for child in node.children:
            if child:
                visualize_tree_levels(child, ax, level, current_level + 1)

# Plot the spatial distribution
size = 8
num_cols = 3  # Number of columns to display in a row
max_levels = 6  # Maximum number of levels
num_rows = (max_levels + num_cols - 1) // num_cols  # Calculate the required number of rows

plt.figure(figsize=(num_cols * size, num_rows * size))  # Adjust overall size

# Display panels for each level of the tree
for level in range(max_levels):
    plt.subplot(num_rows, num_cols, level + 1)
    ax = plt.gca()
    visualize_tree_levels(bh_tree.root, ax, level)  # Display the tree for each level
    plt.scatter([body.position[0] for body in bodies], [body.position[1] for body in bodies], s=100, color='blue', label='Particles')
    plt.xlim(-R, R)
    plt.ylim(-R, R)
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title(f'Tree Division Level {level}')
    plt.axis('equal')

plt.tight_layout()
plt.show()

In [ ]:
import time

# Configuration
Mtot = 10  # Total mass
num_bodies = 1000  # Number of particles
R = 5  # Radius of the disk

# Generate random positions
theta_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # List of theta values to test
calculation_times = []  # List to store calculation times
percentage_errors = []  # List to store errors compared to direct calculations

# Generate random angles theta and distances r for each particle
theta_random = np.random.uniform(0, 2 * np.pi, num_bodies)  # Random angles (0 to 2π)
r = np.sqrt(np.random.uniform(0, R**2, num_bodies))  # Random radii (0 to R)

# Convert from polar to Cartesian coordinates
x_positions = r * np.cos(theta_random)
y_positions = r * np.sin(theta_random)

# Create Body instances (assuming equal mass of Mtot / num_bodies)
mass = Mtot / num_bodies
bodies = [Body(position=(x, y), mass=mass) for x, y in zip(x_positions, y_positions)]

# Directly calculate gravitational forces between all bodies
start_time = time.time()
forces_direct = calculate_gravitational_force_direct(bodies)
time_direct = time.time() - start_time
print(time_direct)

# Create a Barnes-Hut tree
boundary = (-R, R, -R, R)
bh_tree = BarnesHut(boundary)
    
# Insert particles into the tree
for body in bodies:
    bh_tree.insert(body)

# Evaluate calculation times and errors against theta
for theta in theta_values:
    # Measure the calculation time using the tree method
    start_time = time.time()
    forces_tree = []
    for body in bodies:
        force = bh_tree.calculate_gravitational_force(body, theta=theta)
        forces_tree.append(force)
    elapsed_time = time.time() - start_time

    calculation_times.append(elapsed_time)

    # Calculate the differences between the tree method and direct calculation
    forces_diff_magnitudes = [100 * np.linalg.norm(forces_tree[i] - forces_direct[i]) / np.linalg.norm(forces_direct[i]) 
                              for i in range(num_bodies) if np.linalg.norm(forces_direct[i]) != 0]
    
    # Calculate the mean error
    mean_error = np.mean(forces_diff_magnitudes)
    percentage_errors.append(mean_error)

    # Output the results
    print(f"Theta: {theta}, Calculation Time: {elapsed_time:.4f} seconds, {elapsed_time/time_direct: .4f}, Mean Error: {mean_error:.2f} %")

#########################
# Plotting
size = 8
lw = 3      # Line width
msize = 10  # Marker size
plt.figure(figsize=(3 * size, size))

# Left panel
scale_factor = 1
plt.subplot(1, 3, 1)
bh_tree.visualize_tree(bh_tree.root, plt.gca())
plt.scatter([body.position[0] for body in bodies], [body.position[1] for body in bodies], s=100, color='blue', label='Particles')

plt.xlim(-R, R)
plt.ylim(-R, R)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Direct Gravitational Force')
plt.grid()
plt.legend()
plt.axis('equal')

# Plotting calculation times
plt.subplot(1, 3, 2)
plt.plot(theta_values, calculation_times, marker='o', color='blue', linewidth=lw, markersize=msize, label='Calculation Time')
plt.xlabel('Opening Angle $\\theta$')
plt.ylabel('Calculation Time (sec)')
plt.title('Calculation Time vs Theta')
plt.grid(True)

# Plotting mean errors
plt.subplot(1, 3, 3)
plt.plot(theta_values, percentage_errors, marker='o', color='red', linewidth=lw, markersize=msize, label='Mean Error')
plt.xlabel('Opening Angle $\\theta$')
plt.ylabel('Mean Error [%]')
# plt.xscale('log')
plt.yscale('log')
plt.title('Mean Error vs Theta')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# The theta benchmark is performed in the preceding cell.